# Colab Public Face Pretraining

Notebook này chạy public face pretraining theo flow kiểm soát dần:

- `smoke`: xác nhận pipeline hoạt động
- `mid`: xác nhận learning signal
- `e10`: run public pretraining có ý nghĩa đầu tiên
- `full`: run dài hơn để tạo checkpoint seed cho internal finetuning

Lưu ý:
- public dataset hiện dùng `Face_Recognition/train` để script tự random split train/val
- không dùng trực tiếp `Face_Recognition/val` với `train_phase3.py`
- KD và pruning đều tắt ở giai đoạn này


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive', force_remount=True)

DRIVE_ROOT = Path('/content/drive/MyDrive')
PROJECT_ROOT = DRIVE_ROOT / 'Attendance_Workspace' / '3_edgeface_training'
DATASET_ROOT = DRIVE_ROOT / 'Face_Recognition' / 'train'
CHECKPOINT_DIR = PROJECT_ROOT / 'checkpoints'

STAGE_PRESETS = {
    'smoke': {
        'output_prefix': 'phase3_public_pretrain_smoke',
        'epochs': 3,
        'batch_size': 32,
        'num_workers': 2,
        'learning_rate': 1e-4,
        'max_train_batches_per_epoch': 200,
        'max_val_batches': 50,
    },
    'mid': {
        'output_prefix': 'phase3_public_pretrain_mid',
        'epochs': 5,
        'batch_size': 32,
        'num_workers': 2,
        'learning_rate': 1e-4,
        'max_train_batches_per_epoch': 500,
        'max_val_batches': 100,
    },
    'e10': {
        'output_prefix': 'phase3_public_pretrain_e10',
        'epochs': 10,
        'batch_size': 64,
        'num_workers': 2,
        'learning_rate': 1e-4,
        'max_train_batches_per_epoch': 1000,
        'max_val_batches': 200,
    },
    'full': {
        'output_prefix': 'phase3_public_pretrain',
        'epochs': 20,
        'batch_size': 64,
        'num_workers': 2,
        'learning_rate': 1e-4,
        'max_train_batches_per_epoch': 2000,
        'max_val_batches': 300,
    },
}

ACTIVE_STAGE = 'mid'
WIDTH_PRESET = 'widened'
RANK_RATIO = 0.7

cfg = STAGE_PRESETS[ACTIVE_STAGE]
print('ACTIVE_STAGE =', ACTIVE_STAGE)
print('PROJECT_ROOT =', PROJECT_ROOT)
print('DATASET_ROOT =', DATASET_ROOT)
print('CHECKPOINT_DIR =', CHECKPOINT_DIR)
print('STAGE_CONFIG =', cfg)


In [ ]:
assert PROJECT_ROOT.exists(), f'Project root not found: {PROJECT_ROOT}'
assert DATASET_ROOT.exists(), f'Dataset root not found: {DATASET_ROOT}'

train_entries = [p for p in DATASET_ROOT.iterdir() if p.is_dir()]
print('train_shard_count =', len(train_entries))
print('sample_train_entries =', [p.name for p in train_entries[:5]])


In [ ]:
%cd {PROJECT_ROOT}
!pip install -q -r requirements.txt


In [ ]:
import shlex
import subprocess

cmd = [
    'python',
    'scripts/train_phase3.py',
    '--dataset-root', str(DATASET_ROOT),
    '--checkpoints-dir', str(CHECKPOINT_DIR),
    '--epochs', str(cfg['epochs']),
    '--batch-size', str(cfg['batch_size']),
    '--num-workers', str(cfg['num_workers']),
    '--learning-rate', str(cfg['learning_rate']),
    '--width-preset', WIDTH_PRESET,
    '--rank-ratio', str(RANK_RATIO),
    '--kd-alpha', '0',
    '--skip-student-bootstrap',
    '--skip-teacher-bootstrap',
    '--max-train-batches-per-epoch', str(cfg['max_train_batches_per_epoch']),
    '--max-val-batches', str(cfg['max_val_batches']),
    '--output-prefix', cfg['output_prefix'],
]

print('Running command:')
print(' '.join(shlex.quote(part) for part in cmd))
subprocess.run(cmd, check=True, cwd=PROJECT_ROOT)


## Decision Rule After Each Stage

- Sau `smoke`: chỉ cần xác nhận pipeline chạy, loss có xu hướng giảm, checkpoint được ghi ra Drive.
- Sau `mid`: nếu train/val accuracy tăng rõ hơn smoke run và loss còn giảm, chuyển sang `e10`.
- Sau `e10`: nếu validation còn tăng, mới cân nhắc `full`.
- Không bật KD, không pruning, không đổi kiến trúc trong giai đoạn public pretraining.
